# 📊 01: Exploratory Data Analysis & Text Preprocessing Pipeline
### YouTube Comment Sentiment & Audience Intelligence
---
**Objective:** Explore the raw multi-class comment dataset, analyze class distributions, assess text length & vocabulary statistics, handle missing/duplicate entries, and apply production-grade NLP text cleaning.

In [ ]:
# 1. Imports & Configuration
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import DataLoader
from src.data.preprocessor import TextPreprocessor
from src.utils.config_manager import load_config

sns.set_theme(style='whitegrid')
config = load_config('../configs/config.yaml')

In [ ]:
# 2. Ingest Dataset & Validate Schema
loader = DataLoader(config)
df_raw = loader.fetch_raw_data()
print(f'Raw dataset shape: {df_raw.shape}')
df_raw.head()

In [ ]:
# 3. Clean Missing Values, Whitespaces, and Duplicates
df = loader.validate_and_clean_schema(df_raw)
print(f'Cleaned dataset shape: {df.shape}')
df.info()

In [ ]:
# 4. Sentiment Class Distribution
sentiment_map = {-1: 'Negative', 0: 'Neutral', 1: 'Positive'}
df['sentiment_label'] = df['category'].map(sentiment_map)

plt.figure(figsize=(7, 4))
ax = sns.countplot(data=df, x='sentiment_label', palette=['crimson', 'gold', 'forestgreen'], order=['Negative', 'Neutral', 'Positive'])
plt.title('Sentiment Class Distribution in Dataset', fontsize=14, weight='bold')
plt.xlabel('Sentiment Category')
plt.ylabel('Count')
plt.show()

In [ ]:
# 5. Comment Length & Word Count Statistics
df['char_count'] = df['clean_comment'].apply(len)
df['word_count'] = df['clean_comment'].apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(data=df, x='sentiment_label', y='word_count', ax=axes[0], palette=['crimson', 'gold', 'forestgreen'], order=['Negative', 'Neutral', 'Positive'], showfliers=False)
axes[0].set_title('Word Count Distribution per Sentiment (Capped Outliers)')
sns.kdeplot(data=df, x='word_count', hue='sentiment_label', common_norm=False, ax=axes[1], palette=['crimson', 'gold', 'forestgreen'], cut=0)
axes[1].set_xlim(0, 100)
axes[1].set_title('Word Count KDE Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# 6. Apply Production NLP Preprocessor
preprocessor = TextPreprocessor(config)
sample_text = "I don't think this is a good tutorial! Visit https://example.com @user #awesome"
print(f'Original:  {sample_text}')
print(f'Cleaned:   {preprocessor.clean_text(sample_text)}')